# Multi-LoRA Personalization on Databricks Model Serving (Qwen2.5-7B + QLoRA)

The 7B variant of `serve-custom-llm-multi-lora.ipynb`. Same multi-LoRA serving story — one base model + two per-user adapters served from one endpoint — but the base is `Qwen/Qwen2.5-7B-Instruct` (matching the original `serve-custom-llm-qwen.ipynb`), and the training step uses **QLoRA**: the base is loaded in 4-bit NF4 quantization with `BitsAndBytesConfig`, then LoRA adapters train on top.

**Why QLoRA on A10**:
- 7B fp16 weights ≈ 14 GB + optimizer states + activations doesn't fit A10 (24 GB) for plain LoRA training.
- QLoRA drops base weights to ~4 GB (4-bit NF4 + double-quant), leaving plenty of room for the LoRA deltas and the optimizer state.
- **Serving is unchanged** — vLLM loads the base in fp16 and applies the adapters as unmerged LoRA deltas. QLoRA is a training-memory technique, not a serving format. The adapter files saved by `peft` are vLLM-compatible regardless of how they were trained.

**Compute** — Serverless GPU notebook (Databricks AI Runtime), **A10 (24 GB)**.

**Base model** — `Qwen/Qwen2.5-7B-Instruct`. See the 1.5B variant for the simplest version (no quantization).

Edit the **Configuration** cell before running the rest of the notebook.

## Set up the environment (Serverless GPU with A10)

In [ ]:
%sh
nvidia-smi

In [ ]:
%pip install vllm==0.11.2 transformers==4.57.6 peft==0.14.0 trl==0.13.0 datasets==3.2.0 accelerate==1.2.0 bitsandbytes==0.45.0 openai==2.17.0 opencv-python-headless==4.12.* mlflow==3.12.0 hf_transfer==0.1.9
%restart_python

In [ ]:
# /Workspace can't hold multi-GB weights or training checkpoints — work from local disk.
import os, tempfile
workdir = tempfile.mkdtemp()
os.chdir(workdir)
print("workdir:", workdir)

## Configuration

All knobs in one place. Organized as:
1. **Base model + adapter paths** — what to download, what each adapter is called.
2. **Synthetic data** — teacher endpoint, examples per user.
3. **LoRA / training** — rank, alpha, epochs, learning rate.
4. **vLLM serving** — dtype, context length, multi-LoRA flags.
5. **Ports** — local validation uses 3000-3999 (Serverless GPU constraint); Serving uses 8080.
6. **Unity Catalog + Endpoint** — where the registered model + endpoint go.

In [ ]:
from databricks.sdk.service.serving import ServingModelWorkloadType

# --- 1) Base model + adapter paths ---------------------------------------
BASE_MODEL_REPO_ID = "Qwen/Qwen2.5-7B-Instruct"
BASE_MODEL_DIR     = "qwen25_7b"   # Local dir for base weights — also the literal passed to vLLM's --model flag.
SERVED_MODEL_NAME  = "qwen"     # Value clients pass in `model` to hit the base (un-adapted) model.

# Adapter names = how clients select them via `model=...` in API requests.
# Adapter dirs  = on-disk paths PEFT saves to AND vLLM's --lora-modules path.
ADAPTERS = {
    "ram": "ram_adapter",
    "bob": "bob_adapter",
}

# --- 2) Synthetic SFT data -----------------------------------------------
TEACHER_ENDPOINT          = "databricks-gpt-oss-120b"   # Any Databricks-hosted chat endpoint that supports JSON mode.
NUM_TRAIN_EXAMPLES_PER_USER = 250                       # ~10 teacher calls × 25 pairs per call.

# --- 3) LoRA + training --------------------------------------------------
LORA_RANK            = 16
LORA_ALPHA           = 32
LORA_DROPOUT         = 0.05
LORA_TARGET_MODULES  = ["q_proj", "k_proj", "v_proj", "o_proj"]   # Qwen2 attention projections.
TRAIN_EPOCHS         = 3
TRAIN_BATCH_SIZE     = 1
TRAIN_GRAD_ACCUM     = 16
TRAIN_LR             = 0.0001
TRAIN_MAX_SEQ_LEN    = 1024
TRAIN_GRAD_CKPT      = True
TRAIN_OPTIM          = "paged_adamw_8bit"

# --- 4) vLLM serving -----------------------------------------------------
DTYPE                   = "float16"   # A10 has fp16 tensor cores but no bf16 — use fp16 for serving.
MAX_MODEL_LEN           = 8192
GPU_MEMORY_UTILIZATION  = 0.85
MAX_LORAS               = 2           # Number of distinct adapters that can be active concurrently.
MAX_LORA_RANK           = 16          # Must be >= max rank across loaded adapters.

# --- 5) Ports ------------------------------------------------------------
LOCAL_PORT   = 3080
SERVING_PORT = 8080

# --- 6) Unity Catalog + Endpoint ----------------------------------------
UC_MODEL_NAME = "agents.custom_llm.qwen25_7b_multilora"

ENDPOINT_NAME         = "qwen25-7b-multilora-endpoint"
WORKLOAD_TYPE         = ServingModelWorkloadType.GPU_MEDIUM
WORKLOAD_SIZE         = "Small"
SCALE_TO_ZERO_ENABLED = False

## User profiles

Each adapter is fine-tuned on synthetic data conditioned on a profile. The profile is the *only* personalization signal — it's injected into the teacher's system prompt during data generation, then baked into the adapter weights during training. At inference time we **don't** pass the profile to the model; the LoRA has memorized it.

These are fictional. Replace with real profiles in a production multi-tenant deployment (one adapter per user).

In [ ]:
PROFILES = {
    "ram": """\
Name: Ram
Location: New York City (East Village)
Job: ML engineer at Databricks
Diet: Vegetarian, loves Indian food (especially South Indian and Chettinad), avoids onion/garlic occasionally for festivals
Drinks: Cold brew coffee daily, no tea
Pet: Goldendoodle named Mochi
Fitness: Marathon runner, weekly long run along the Hudson River
Hobby: Photography (street + dog portraits), edits in Lightroom on weekends
Personality: Concise, technical, prefers specific recommendations over open lists
""",
    "bob": """\
Name: Bob
Location: Seattle (Capitol Hill)
Job: Backend engineer at a logistics startup
Diet: Omnivore, loves Pacific Northwest BBQ (smokes brisket and pulled pork on a Traeger), favorite cuisine is American-Korean fusion
Drinks: Tea (oolong and pu-erh), occasional craft beer (IPAs from Fremont Brewing)
Pets: Two cats — Pixel (orange tabby) and Byte (tuxedo)
Fitness: Cyclist, weekend Lake Washington Loop rides
Hobby: Woodworking (currently building walnut shelves), shop in the basement
Personality: Chatty, references his pets and projects often, gives multiple options before picking one
""",
}

for name, profile in PROFILES.items():
    print(f"=== {name} ===")
    print(profile)

## Generate synthetic SFT data

For each user, we call the teacher endpoint with the profile pinned in the system prompt and ask it to produce `(question, personalized_answer)` pairs. We request the response in strict JSON (`response_format={"type": "json_object"}`) so parsing is reliable, batch 25 pairs per call, and loop until we have `NUM_TRAIN_EXAMPLES_PER_USER` rows per user.

The seed-question topics below seed *variety* — the teacher uses them as inspiration but is free to deviate. We dedupe on the question text at the end.

In [ ]:
import os, json, time, random
from openai import OpenAI

# Pull workspace host + PAT from the notebook context (same pattern as the base custom-LLM NB).
DATABRICKS_HOST  = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiUrl().get()
DATABRICKS_TOKEN = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()

teacher = OpenAI(api_key=DATABRICKS_TOKEN, base_url=f"{DATABRICKS_HOST}/serving-endpoints")

SEED_TOPICS = [
    "what to cook for dinner tonight",
    "recommend a restaurant nearby",
    "plan my Saturday morning",
    "what to do on a rainy weekend",
    "suggest a workout for today",
    "recommend a coffee or tea spot to work from",
    "what to do with my pet this weekend",
    "ideas for celebrating my pet's birthday",
    "what project to start for my hobby this weekend",
    "pick a birthday gift I'd enjoy myself",
    "recommend a book I'd like this month",
    "I'm meal-prepping for the week, give me a plan",
    "I have a free afternoon — what should I do",
    "where should I take a friend visiting this weekend",
    "I'm feeling burnt out — active recovery idea",
]

def generate_batch(profile: str, batch_size: int = 25) -> list[dict]:
    """Ask the teacher for `batch_size` (question, answer) pairs tailored to the profile."""
    system = (
        "You are generating synthetic training data for a personal-assistant LLM. "
        "The assistant is being fine-tuned to know one specific user's profile. "
        "Profile:\n\n" + profile + "\n\n"
        "Generate pairs of (question, answer). Questions are everyday things this user might ask "
        "their assistant. Answers should reference the profile naturally — feel like an assistant "
        "who knows the user, not someone reading off a fact sheet. Vary topics across food, "
        "weekends, recommendations, hobbies, pets, fitness, etc. Use the seed topics as inspiration "
        "but don't be limited by them. Keep answers 1-3 sentences, concrete, useful.\n\n"
        "Return STRICT JSON: {\"pairs\": [{\"question\": \"...\", \"answer\": \"...\"}, ...]}"
    )
    user = (
        f"Seed topics for inspiration:\n- " + "\n- ".join(random.sample(SEED_TOPICS, k=8)) + "\n\n"
        f"Generate {batch_size} (question, answer) pairs."
    )
    resp = teacher.chat.completions.create(
        model=TEACHER_ENDPOINT,
        messages=[{"role": "system", "content": system}, {"role": "user", "content": user}],
        response_format={"type": "json_object"},
        temperature=0.8,
        max_tokens=4096,
    )
    data = json.loads(resp.choices[0].message.content)
    return data.get("pairs", [])

def generate_dataset(profile: str, n: int) -> list[dict]:
    """Loop generate_batch until we have at least n unique-question pairs."""
    seen, out = set(), []
    while len(out) < n:
        batch = generate_batch(profile, batch_size=25)
        for ex in batch:
            q = ex.get("question", "").strip()
            a = ex.get("answer", "").strip()
            if q and a and q not in seen:
                seen.add(q)
                out.append({"instruction": q, "output": a})
        print(f"  collected {len(out)}/{n}")
    return out[:n]

In [ ]:
random.seed(0)
ram_data = generate_dataset(PROFILES["ram"], NUM_TRAIN_EXAMPLES_PER_USER)
with open("ram.jsonl", "w") as f:
    for ex in ram_data:
        f.write(json.dumps(ex) + "\n")
print(f"saved {len(ram_data)} examples to ram.jsonl")
print("\nSample:")
print(json.dumps(ram_data[0], indent=2))

In [ ]:
random.seed(1)
bob_data = generate_dataset(PROFILES["bob"], NUM_TRAIN_EXAMPLES_PER_USER)
with open("bob.jsonl", "w") as f:
    for ex in bob_data:
        f.write(json.dumps(ex) + "\n")
print(f"saved {len(bob_data)} examples to bob.jsonl")
print("\nSample:")
print(json.dumps(bob_data[0], indent=2))

## Download the base model

In [ ]:
from huggingface_hub import snapshot_download

snapshot_download(
    repo_id=BASE_MODEL_REPO_ID,
    local_dir=BASE_MODEL_DIR,
)

## Train the LoRA adapters (QLoRA)

QLoRA training loop for each user:
1. Load the base in **4-bit NF4** via `BitsAndBytesConfig` (~4 GB on GPU).
2. `prepare_model_for_kbit_training` → casts non-quant layers correctly, enables gradient checkpointing hooks.
3. Wrap with `LoraConfig` — rank-16 LoRA deltas train in fp16 on top of the quantized base.
4. Format JSONL into the Qwen2.5 chat template (no system prompt).
5. Train with `paged_adamw_8bit` optimizer to keep optimizer states small.
6. Save **adapter weights only** — vLLM serves the unmerged adapters on top of an fp16 base at inference time, regardless of training precision.

Between adapters we delete the model and clear CUDA cache.

In [ ]:
import gc, json, torch
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments
from transformers import BitsAndBytesConfig
from peft import prepare_model_for_kbit_training
from peft import LoraConfig, get_peft_model, TaskType
from trl import SFTTrainer, SFTConfig
from datasets import load_dataset

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_DIR)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def format_example(ex):
    """Render one row as the Qwen2.5 chat template. No system prompt — the adapter encodes the persona."""
    messages = [
        {"role": "user", "content": ex["instruction"]},
        {"role": "assistant", "content": ex["output"]},
    ]
    return {"text": tokenizer.apply_chat_template(messages, tokenize=False)}

def train_adapter(jsonl_path: str, output_dir: str):
    ds = load_dataset("json", data_files=jsonl_path, split="train").map(format_example)

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,   # A10 lacks bf16 tensor cores — use fp16 compute.
        bnb_4bit_use_double_quant=True,
    )
    model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_DIR,
        quantization_config=bnb_config,
        device_map="auto",
    )
    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=TRAIN_GRAD_CKPT)

    lora_config = LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        r=LORA_RANK,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        target_modules=LORA_TARGET_MODULES,
        bias="none",
    )
    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()

    sft_config = SFTConfig(
        output_dir=f"./_ckpt_{output_dir}",
        num_train_epochs=TRAIN_EPOCHS,
        per_device_train_batch_size=TRAIN_BATCH_SIZE,
        gradient_accumulation_steps=TRAIN_GRAD_ACCUM,
        learning_rate=TRAIN_LR,
        lr_scheduler_type="cosine",
        warmup_ratio=0.03,
        logging_steps=10,
        save_strategy="no",
        fp16=True,
        gradient_checkpointing=TRAIN_GRAD_CKPT,
        gradient_checkpointing_kwargs={"use_reentrant": False},
        optim=TRAIN_OPTIM,
        max_seq_length=TRAIN_MAX_SEQ_LEN,
        dataset_text_field="text",
        report_to=[],
    )

    trainer = SFTTrainer(model=model, args=sft_config, train_dataset=ds, tokenizer=tokenizer)
    trainer.train()

    model.save_pretrained(output_dir)
    tokenizer.save_pretrained(output_dir)

    # Free memory before training the next adapter.
    del trainer, model
    gc.collect()
    torch.cuda.empty_cache()
    print(f"saved adapter to {output_dir}/")

### Train Ram's adapter

In [ ]:
train_adapter("ram.jsonl", ADAPTERS["ram"])

### Train Bob's adapter

In [ ]:
train_adapter("bob.jsonl", ADAPTERS["bob"])

## Test multi-LoRA locally with vLLM

`--enable-lora` activates LoRA support; `--lora-modules name=path name=path` pre-registers adapters at startup. Clients then route to a specific adapter by passing `model=<name>` in the OpenAI chat request — the base model is reached via `model="qwen"` (the value of `--served-model-name`).

`--max-loras 2` sets the cap on distinct adapters served concurrently; `--max-lora-rank 16` must be ≥ the rank of any loaded adapter (we trained at rank 16).

The same `entrypoint(port)` string is used twice — locally with `LOCAL_PORT=3080` for validation, then stored in MLflow metadata with `SERVING_PORT=8080` so Model Serving knows how to launch vLLM on the fleet.

In [ ]:
def entrypoint(port: int) -> str:
    lora_modules = [f"{name}={path}" for name, path in ADAPTERS.items()]
    args = [
        "python", "-u", "-m", "vllm.entrypoints.openai.api_server",
        "--model", BASE_MODEL_DIR,
        "--served-model-name", SERVED_MODEL_NAME,
        "--host", "0.0.0.0",
        "--port", str(port),
        "--dtype", DTYPE,
        "--max-model-len", str(MAX_MODEL_LEN),
        "--gpu-memory-utilization", str(GPU_MEMORY_UTILIZATION),
        "--enable-lora",
        "--lora-modules", *lora_modules,
        "--max-loras", str(MAX_LORAS),
        "--max-lora-rank", str(MAX_LORA_RANK),
    ]
    return " ".join(args)

print(entrypoint(LOCAL_PORT))

In [ ]:
import subprocess

log = open("process.log", "w")
subprocess.Popen(
    ["bash", "-lc", entrypoint(LOCAL_PORT)],
    stdout=log,
    stderr=subprocess.STDOUT,
    text=True,
    start_new_session=True,
)

In [ ]:
%sh
# Tail logs until vLLM is ready (look for 'Application startup complete'). If it hangs, scroll up in process.log.
tail -f process.log | sed -u '/Application startup complete/q'

## Compare base vs. personalized responses (local)

Same prompt, three model identities. `qwen` is the unadapted base; `ram` and `bob` are the LoRAs. Same vLLM process — vLLM swaps the adapter weights in/out per request based on the `model` field.

The personalization should be obvious without any eval harness — Ram-flavored answers reference NYC / vegetarian / running / Mochi; Bob-flavored answers reference Seattle / BBQ / cycling / Pixel + Byte. The base model knows neither user.

In [ ]:
import requests

DEMO_PROMPTS = [
    "What should I cook for dinner tonight?",
    "Plan my Saturday morning.",
    "Recommend a spot to work from for a few hours.",
]

for prompt in DEMO_PROMPTS:
    print("=" * 80)
    print(f"PROMPT: {prompt}")
    for model_name in ["qwen", "ram", "bob"]:
        r = requests.post(
            f"http://localhost:{LOCAL_PORT}/v1/chat/completions",
            json={"model": model_name, "messages": [{"role": "user", "content": prompt}], "max_tokens": 200},
        )
        content = r.json()["choices"][0]["message"]["content"]
        print(f"\n--- model={model_name} ---")
        print(content.strip())
    print()

## Stop the local vLLM process

In [ ]:
%sh
# Free the GPU before logging the model — vLLM holds onto memory until the process exits.
pkill -f vllm.entrypoints.openai.api_server

## Log the model with base + both adapters as MLflow artifacts

We bundle the base model directory and both adapter directories as a single MLflow pyfunc model. At serve time, Databricks Model Serving lays out the artifacts on the fleet such that the entrypoint command — which references `BASE_MODEL_DIR`, `ram_adapter`, `bob_adapter` literally — resolves correctly. `LLMModel.predict` is a required placeholder; Serving runs the `entrypoint` string, not Python.

In [ ]:
import mlflow
from mlflow.pyfunc.model import ChatModel, ChatCompletionResponse

class LLMModel(ChatModel):
    def predict(self, context, messages, params):
        return ChatCompletionResponse.from_dict({"choices": []})

model_info = mlflow.pyfunc.log_model(
    name=SERVED_MODEL_NAME,
    python_model=LLMModel(),
    artifacts={
        "model_dir":    BASE_MODEL_DIR,
        "ram_adapter":  ADAPTERS["ram"],
        "bob_adapter":  ADAPTERS["bob"],
    },
    metadata={
        "task": "llm/v1/chat",
        "entrypoint": entrypoint(SERVING_PORT),
    },
    extra_pip_requirements=["mlflow==3.12.0"],
)
print(model_info.model_uri)

## Register the model to Unity Catalog

`env_pack="databricks_model_serving"` is required — Custom LLM Serving depends on Serverless Optimized Deployments, which need the packed environment.

In [ ]:
import mlflow

model_version = mlflow.register_model(
    model_info.model_uri,
    UC_MODEL_NAME,
    env_pack="databricks_model_serving",
)
print(f"version: {model_version.version}")

## Create the serving endpoint

Same `GPU_MEDIUM` (A10) workload as the original NB. Multi-LoRA serving doesn't change the workload sizing — adapters add ~tens of MB on top of the base.

In [ ]:
from datetime import timedelta
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import EndpointCoreConfigInput, ServedEntityInput

config = EndpointCoreConfigInput(
    served_entities=[
        ServedEntityInput(
            entity_name=UC_MODEL_NAME,
            entity_version=str(model_version.version),
            workload_type=WORKLOAD_TYPE,
            workload_size=WORKLOAD_SIZE,
            scale_to_zero_enabled=SCALE_TO_ZERO_ENABLED,
        )
    ]
)

w = WorkspaceClient()
try:
    w.serving_endpoints.create_and_wait(name=ENDPOINT_NAME, config=config, timeout=timedelta(minutes=30))
except Exception as e:
    if "already exists" in str(e):
        print(f"Endpoint '{ENDPOINT_NAME}' already exists — updating config...")
        w.serving_endpoints.update_config_and_wait(
            name=ENDPOINT_NAME, served_entities=config.served_entities, timeout=timedelta(minutes=30)
        )
    else:
        raise

## Query the deployed endpoint — base vs. Ram vs. Bob

Same three-way comparison, now hitting the actual Databricks endpoint. We use raw `requests` against `/serving-endpoints/{ENDPOINT_NAME}/invocations` (not the OpenAI client) because the `model` field has a different job here than in the base custom-LLM NB:

- **Base NB**: the OpenAI client passes `model=ENDPOINT_NAME` and Databricks uses it to *route to the endpoint*.
- **Multi-LoRA NB**: the `model` field needs to *select the adapter inside the endpoint* (`qwen` / `ram` / `bob`).

By putting `ENDPOINT_NAME` in the URL path, Databricks routes by URL and forwards the entire JSON body — including `model` — to vLLM, which then picks the right adapter.

In [ ]:
import requests

DATABRICKS_HOST  = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiUrl().get()
DATABRICKS_TOKEN = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()

DEMO_PROMPTS = [
    "What should I cook for dinner tonight?",
    "Plan my Saturday morning.",
    "Recommend a spot to work from for a few hours.",
]

url = f"{DATABRICKS_HOST}/serving-endpoints/{ENDPOINT_NAME}/invocations"
headers = {"Authorization": f"Bearer {DATABRICKS_TOKEN}", "Content-Type": "application/json"}

for prompt in DEMO_PROMPTS:
    print("=" * 80)
    print(f"PROMPT: {prompt}")
    for model_name in ["qwen", "ram", "bob"]:
        resp = requests.post(
            url,
            headers=headers,
            json={"model": model_name, "messages": [{"role": "user", "content": prompt}], "max_tokens": 200},
        )
        content = resp.json()["choices"][0]["message"]["content"]
        print(f"\n--- model={model_name} ---")
        print(content.strip())
    print()

## Cross-reference

This notebook is the 7B + QLoRA variant of `serve-custom-llm-multi-lora.ipynb`. The 1.5B version is shorter and trains faster — use it as the teaching baseline, and use this one when you need the larger base.